<a href="https://colab.research.google.com/github/ImNotDanish05/2026_Pembelajaran-Mesin_TI-3A/blob/main/Week_8/FiturSelection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Unduh dataset menggunakan wget
!wget "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip" -O diabetes_dataset.zip

--2026-10-20 01:43:13--  https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘diabetes_dataset.zip’

diabetes_dataset.zi     [   <=>              ]   3.02M  6.71MB/s    in 0.5s    

2026-10-20 01:43:13 (6.71 MB/s) - ‘diabetes_dataset.zip’ saved [3170254]



In [2]:
# Ekstrak file zip yang telah diunduh
!unzip -o diabetes_dataset.zip -d diabetes_data

Archive:  diabetes_dataset.zip
  inflating: diabetes_data/diabetic_data.csv  
  inflating: diabetes_data/IDS_mapping.csv  


Directory Dataset: <br><br>/content/diabetes_data/diabetic_data.csv

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 1. Load Dataset
df = pd.read_csv('/content/diabetes_data/diabetic_data.csv')

# Tampilkan informasi dasar data
print(f"Dimensi awal dataset: {df.shape}")
display(df.head())


Dimensi awal dataset: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [4]:
# 2. Preprocessing Data
# Mengganti karakter '?' dengan NaN
df = df.replace('?', np.nan)

# Mengurangi beban memori/komputasi dengan menghapus kolom dengan terlalu banyak missing values
# dan kolom identifikasi yang tidak relevan untuk prediksi
kolom_dihapus = ['encounter_id', 'patient_nbr', 'weight', 'payer_code', 'medical_specialty']
df = df.drop(columns=[col for col in kolom_dihapus if col in df.columns])

# Mengisi missing values sederhana untuk fitur kategorikal & numerik sisanya
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# Target variabel (klasifikasi): readmitted
# Kita akan memprediksi apakah pasien akan kembali dirawat ('readmitted')
target_col = 'readmitted'

# Label Encoding untuk kolom objek (kategorikal)
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col].astype(str))

# Memisahkan Fitur (X) dan Target (y)
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"Dimensi fitur setelah preprocessing: {X.shape}")


Dimensi fitur setelah preprocessing: (101766, 44)


In [5]:
from sklearn.feature_selection import VarianceThreshold

# Memisahkan fitur sebelum standarisasi
# Kita akan melihat varians dari masing-masing fitur
# Kita tentukan threshold, misalnya threshold = 0.1 (fitur yang variansnya di bawah 10% akan dibuang)
# Karena sebagian besar fitur kategorikal sudah di-encode, kita bisa mengukur variansnya langsung
threshold_val = 0.1
selector = VarianceThreshold(threshold=threshold_val)

# Fit selector pada fitur
X_selected = selector.fit_transform(X)

# Mendapatkan kolom-kolom yang lolos seleksi
fitur_terpilih = X.columns[selector.get_support()]
fitur_terbuang = X.columns[~selector.get_support()]

print(f"Jumlah fitur sebelum Variance Threshold: {X.shape[1]}")
print(f"Jumlah fitur setelah Variance Threshold ({threshold_val}): {X_selected.shape[1]}\n")
print(f"Fitur yang dibuang karena varians rendah (< {threshold_val}):\n{list(fitur_terbuang)}")


Jumlah fitur sebelum Variance Threshold: 44
Jumlah fitur setelah Variance Threshold (0.1): 23

Fitur yang dibuang karena varians rendah (< 0.1):
['max_glu_serum', 'A1Cresult', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']


In [6]:
# 3b. Memisahkan dan Menstandarisasi kembali menggunakan Fitur yang Terpilih
X_train, X_test, y_train, y_test = train_test_split(X[fitur_terpilih], y, test_size=0.2, random_state=42, stratify=y)

# Skala ulang fitur terpilih
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Pembagian dan standarisasi ulang dengan fitur terseleksi selesai.")


Pembagian dan standarisasi ulang dengan fitur terseleksi selesai.


In [7]:
# 4b. Melatih ulang model KNN menggunakan Fitur yang Terseleksi
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# 5b. Evaluasi Ulang Model
y_pred = knn.predict(X_test_scaled)

print("=== Hasil Evaluasi KNN Classifier dengan Variance Threshold ===\n")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))


=== Hasil Evaluasi KNN Classifier dengan Variance Threshold ===

Accuracy Score: 0.5029

Classification Report:
              precision    recall  f1-score   support

           0       0.18      0.12      0.14      2272
           1       0.40      0.40      0.40      7109
           2       0.60      0.65      0.62     10973

    accuracy                           0.50     20354
   macro avg       0.40      0.39      0.39     20354
weighted avg       0.49      0.50      0.49     20354

